# Extract Data

### 1. Accessing Flight Data

Connecting to the database,

In [1]:
# Importing,
from sqlalchemy import create_engine, MetaData, Table, Column, String, Integer

# Creating the engine,
engine = create_engine("sqlite:///flight_data.db", echo=False)
connection = engine.connect()
metadata = MetaData()

# Creating table,
flights = Table(
    "flights",
    metadata,
    Column("FL_DATE", String),
    Column("OP_CARRIER", String),
    Column("OP_CARRIER_FL_NUM", Integer),
    Column("ORIGIN", String),
    Column("DEST", String),
    Column("CRS_DEP_TIME", String),
    Column("DEP_TIME", String),
    Column("DEP_DELAY", Integer),
    Column("TAXI_OUT", Integer),
    Column("WHEELS_OFF", String),
    Column("WHEELS_ON", String),
    Column("TAXI_IN", Integer),
    Column("CRS_ARR_TIME", String),
    Column("ARR_TIME", String),
    Column("ARR_DELAY", Integer),
    Column("CANCELLED", Integer),
    Column("DIVERTED", Integer),
    Column("CRS_ELAPSED_TIME", Integer),
    Column("ACTUAL_ELAPSED_TIME", Integer),
    Column("AIR_TIME", Integer),
    Column("DISTANCE", Integer),
    Column("TOTAL_DELAY", Integer)
)

Creating the select statement,

In [3]:
select_statement = flights.select().where((flights.c.ORIGIN == "JFK") & (flights.c.TOTAL_DELAY > 0))
result = connection.execute(select_statement)

The resulting rows represent delayed flights that only originated from JFK airport. To double check, we explicitly print a few of them out,

In [4]:
preview_rows = connection.execute(select_statement.limit(10)).mappings()
for row in preview_rows:
    print(row)

{'FL_DATE': '2018-01-01', 'OP_CARRIER': '9E', 'OP_CARRIER_FL_NUM': 4026, 'ORIGIN': 'JFK', 'DEST': 'CHS', 'CRS_DEP_TIME': '13:55', 'DEP_TIME': '13:48', 'DEP_DELAY': -7, 'TAXI_OUT': 37, 'WHEELS_OFF': '14:25', 'WHEELS_ON': '16:17', 'TAXI_IN': 11, 'CRS_ARR_TIME': '16:19', 'ARR_TIME': '16:28', 'ARR_DELAY': 9, 'CANCELLED': 0, 'DIVERTED': 0, 'CRS_ELAPSED_TIME': 144, 'ACTUAL_ELAPSED_TIME': 160, 'AIR_TIME': 112, 'DISTANCE': 636, 'TOTAL_DELAY': 16}
{'FL_DATE': '2018-01-01', 'OP_CARRIER': '9E', 'OP_CARRIER_FL_NUM': 4032, 'ORIGIN': 'JFK', 'DEST': 'RDU', 'CRS_DEP_TIME': '11:40', 'DEP_TIME': '11:48', 'DEP_DELAY': 8, 'TAXI_OUT': 32, 'WHEELS_OFF': '12:20', 'WHEELS_ON': '13:39', 'TAXI_IN': 4, 'CRS_ARR_TIME': '13:31', 'ARR_TIME': '13:43', 'ARR_DELAY': 12, 'CANCELLED': 0, 'DIVERTED': 0, 'CRS_ELAPSED_TIME': 111, 'ACTUAL_ELAPSED_TIME': 115, 'AIR_TIME': 79, 'DISTANCE': 427, 'TOTAL_DELAY': 4}
{'FL_DATE': '2018-01-01', 'OP_CARRIER': '9E', 'OP_CARRIER_FL_NUM': 4038, 'ORIGIN': 'JFK', 'DEST': 'BUF', 'CRS_DEP_TIM

### 2. Creating New Database

We need a new engine to create the database,

In [5]:
# Creating the engine,
second_engine = create_engine("sqlite:///jfk_delayed_flights.db", echo=False)
second_metadata = MetaData()

# Creating table,
delayed_flights = Table(
    "delayed_flights",
    second_metadata,
    Column("FL_DATE", String),
    Column("OP_CARRIER", String),
    Column("OP_CARRIER_FL_NUM", Integer),
    Column("ORIGIN", String),
    Column("DEST", String),
    Column("CRS_DEP_TIME", String),
    Column("DEP_TIME", String),
    Column("DEP_DELAY", Integer),
    Column("TAXI_OUT", Integer),
    Column("WHEELS_OFF", String),
    Column("WHEELS_ON", String),
    Column("TAXI_IN", Integer),
    Column("CRS_ARR_TIME", String),
    Column("ARR_TIME", String),
    Column("ARR_DELAY", Integer),
    Column("CANCELLED", Integer),
    Column("DIVERTED", Integer),
    Column("CRS_ELAPSED_TIME", Integer),
    Column("ACTUAL_ELAPSED_TIME", Integer),
    Column("AIR_TIME", Integer),
    Column("DISTANCE", Integer),
    Column("TOTAL_DELAY", Integer)
)

second_metadata.create_all(second_engine)

Now we insert the data in batches,

In [6]:
BATCH_SIZE = 5000
selected_rows = connection.execute(select_statement).mappings()
insert_statement = delayed_flights.insert()
rows_inserted = 0

with second_engine.begin() as destination:
    while True:
        batch = selected_rows.fetchmany(BATCH_SIZE)
        if not batch:
            break

        destination.execute(insert_statement, batch)
        rows_inserted += len(batch)

print(f"Inserted {rows_inserted} Rows.")

Inserted 30751 Rows.
